In [4]:
!pip install pandas textblob matplotlib

In [5]:
import pandas as pd
import os
from textblob import TextBlob
import matplotlib.pyplot as plt

if not os.path.exists("public_opinions"):
    os.mkdir("public_opinions")
if not os.path.exists("piecharts"):
    os.mkdir("piecharts")
if not os.path.exists("probabilities"):
    os.mkdir("probabilities")

In [7]:
# 1. Load the main dataset
main_df = pd.read_csv("candidate_data/election_insight_final.csv")

In [8]:
# 2. Identify all constituencies
constituencies = main_df["Constituency"].unique()

# 3. Create individual data files for each constituency
for const in constituencies:
    main_df[main_df["Constituency"] == const].to_csv(
        f"public_opinions/{const}_data.csv", index=False
    )

In [9]:
# 4. Loop through each file to calculate probabilities and generate charts
for const in constituencies:
    # Read the specific constituency file
    df_const = pd.read_csv(f"public_opinions/{const}_data.csv")

    # Calculate sentiment polarity (using TextBlob)
    df_const["Sentiment"] = df_const["Text"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    # Group data by Candidate
    cand_stats = (
        df_const.groupby("Candidate")
        .agg(Mentions=("Text", "count"), Avg_Sentiment=("Sentiment", "mean"))
        .reset_index()
    )

    # Calculation Logic:
    # A. Normalize sentiment score from [-1, 1] range to [0, 1]
    cand_stats["Sentiment_Norm"] = (cand_stats["Avg_Sentiment"] + 1) / 2
    # B. Calculate Share of Voice (Candidate mentions / Total mentions)
    cand_stats["Share_of_Voice"] = cand_stats["Mentions"] / cand_stats["Mentions"].sum()
    # C. Calculate Raw Score (70% Sentiment + 30% Share of Voice)
    cand_stats["Raw_Score"] = (cand_stats["Sentiment_Norm"] * 0.70) + (
        cand_stats["Share_of_Voice"] * 0.30
    )
    # D. Final Probability (Normalize so the sum for the constituency is exactly 1.0)
    cand_stats["Probability"] = cand_stats["Raw_Score"] / cand_stats["Raw_Score"].sum()

    # 5. Save the specific probabilities to a new CSV file
    cand_stats[["Candidate", "Probability"]].to_csv(
        f"probabilities/{const}_probabilities.csv", index=False
    )

    # 6. Generate and save the Pie Chart
    plt.figure(figsize=(7, 7))
    plt.pie(
        cand_stats["Probability"],
        labels=cand_stats["Candidate"],
        autopct=lambda p: "{:.3f}".format(p * 0.01),  # Display decimals [0, 1]
        startangle=140,
        colors=plt.cm.Set3.colors,
    )
    plt.title(f"Win Probability - {const}")
    plt.savefig(f"piecharts/{const}_pie_chart.png")
    plt.close()

In [10]:
def remove_dir():
    directory_path = "path/to/your/empty_directory"

    try:
        os.rmdir(directory_path)
        # or Path(directory_path).rmdir()
        print(f"Directory '{directory_path}' removed successfully.")
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
    except OSError as e:
        print(f"Error removing directory '{directory_path}': {e}")

# uncomment the line below and run this cell to remove directories
# remove_dir()